# Stage 2, Version 2: Testing Factors Together

**PG-S2-36 - Automatic Discovery of Interpretable Financial Factors**

Abu Wakkas Bastob

Last week's loop judged every factor **on its own**: a factor was kept only if it helped by itself. Dr Liu pointed out (20 Oct) that this throws away factors that only work **together with other factors**. He asked for:

1. **More detailed feedback to Claude** for every factor: the formula, how it did in each year, Logistic Regression vs Random Forest, kept or rejected, how similar it is to other factors, and a short reason why
2. **A team trial**: test each new factor on top of the factors already selected, not only on its own
3. **Pair trials** for promising factors that may work well together
4. **Telling Claude which factors work together**, and asking it for partners
5. **A comparison with last week's one-at-a-time approach**

In his 21 Oct email he also suggested generating a large set of factors and using **feature selection** to find a good combination (Step 11).

This notebook shows the new version, what the results taught me about the data (Steps 9 and 10), and the comparison (Step 12). The new code is written out in the cells below. The parts that did not change since last week (the formula language, the information about the data, the call to Claude and the final test) are imported, and are shown in full in `Stage 2_Demo.ipynb`.

In [ ]:
# Section 1: Setup, the data, and the three runs I compare
import sys, json, re, time, random, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '.')                 # my Stage 1 code in src/ (features, split, models, backtest)
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 110)

from src.features import build_stage1_features
# unchanged since 15 Oct (shown in full in 'Stage 2_Demo.ipynb'):
from src.stage2.formula import FormulaError, INPUTS, WINDOWS, evaluate, operator_guide, validate, wide_prices
from src.stage2.context import BASE_FORMULAS, market_summary, stage1_importance
from src.stage2.llm import ClaudeProposer, RandomProposer, dump
from src.stage2.final_test import run_final_test

raw = pd.read_csv('data/processed/aligned_daily_ohlcv_5y.csv')   # same data as Stage 1
features = build_stage1_features(raw)                              # my 22 Stage 1 factors + the target
prices = wide_prices(raw)

RUN_A = Path('results/stage2/runs/20261015_1635_claude')           # A: last week, every factor judged alone
V2_RUNS = sorted(Path('results/stage2/runs_v2').glob('*'))
RUN_B = [r for r in V2_RUNS if '_replay_' in r.name][-1]            # B: last week's 48 factors, judged the new way
RUN_C = [r for r in V2_RUNS if r.name.endswith('_claude') and '_replay_' not in r.name][-1]      # C: the new loop with Claude

a_board = pd.read_csv(RUN_A / 'scoreboard.csv').fillna({'reason': '', 'closest': ''})
a_kept = json.loads((RUN_A / 'kept_factors.json').read_text())
print('A:', RUN_A.name, '|', len(a_board), 'factors tried |', len(a_kept), 'kept')
for name, run in [('B', RUN_B), ('C', RUN_C)]:
    n = len(pd.read_csv(run / 'scoreboard.csv'))
    team = json.loads((run / 'kept_factors.json').read_text())
    print(f'{name}:', run.name, '|', n, 'factors tried |', len(team), 'in the selected team')

## Step 2: What Changed

| | Last week (A) | This week (C) |
|---|---|---|
| What Claude sees | a scoreboard line per factor: average gain, years helped | a **report card** per factor: gain in each year, LR vs RF, similarity, kept or not and why, and its own lesson |
| How a factor is tested | **alone**: added to my 22 factors | alone, then **on top of the selected team**, and **in pairs** |
| Who gets kept | factors that help alone | factors that make the team better, alone or as a pair |
| Weak-alone factors | thrown away | kept on a **promising list** for pair trials |
| What Claude is asked for | 3 variants + 3 new ideas | 2 **improve** + 2 **new** + 2 **partner** factors, plus a lesson for every factor of the last round |

Everything else is the same as last week: the formula language (past data only), the four years 2022-2025 (train on Q1-Q2, check on Q3), Logistic Regression and Random Forest, and Q4 is only used once, in the final test.

**One round of the new loop:**
```
Claude proposes 6 factors (2 improve, 2 new, 2 partner)
        |
check the formula and compute it (past data only)
        |
1. alone:        added to my 22 factors                      (same as last week)
2. team trial:   added to my 22 factors + the selected team  (new)
3. pair trials:  two promising factors added together        (new)
        |
joins the team  /  stays on the promising list  /  rejected
        |
a report card for every factor goes back to Claude in the next round
```

## Step 3: The Report Card Claude Gets

Every factor Claude has tried gets a few lines like the ones below. The "why" line is written automatically from the numbers, so it never contains anything that wasn't measured. In the next round Claude has to write its own one-line lesson about each factor, and that lesson is shown on the card from then on.

In [ ]:
# Section 3a: The report card  (also saved as src/stage2/feedback.py)
#
# The report card Claude sees for every factor it has tried (Liu, 20 Oct).
#
# For each factor:
#   - the formula
#   - the change in validation AUC in each year (2022-2025) and the average
#   - whether it helped Logistic Regression and/or Random Forest
#   - how similar it is to the closest existing factor
#   - kept, promising or rejected, and the reason
#   - a short automatic "why" line written from the numbers
#   - Claude's own one-line lesson about it (written in the following round)
#   - its results in the team trial and pair trials (when it had them)
#
# Everything here only formats numbers that were already measured on the
# validation quarters (Q3). Nothing from the Q4 test quarters is ever shown.

import math

YEARS = (2022, 2023, 2024, 2025)
TEXT_FIELDS = ("name", "mode", "parent", "role", "formula", "hypothesis", "status", "reason", "lesson",
               "how_selected", "group", "closest", "best_pair_partner")
WHOLE_FIELDS = ("round", "years_helped", "team_size", "team_years_helped", "best_pair_years")


def _num(x):
    try:
        x = float(x)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(x) else x


def board_records(board) -> list:
    """Rows of a saved scoreboard.csv (a pandas table) as the records used for the report cards and the loop."""
    out = []
    for r in board.to_dict("records"):
        r = {k: v for k, v in r.items() if not (isinstance(v, float) and math.isnan(v))}
        r.update({k: str(r.get(k, "")) for k in TEXT_FIELDS})
        r.update({k: int(r[k]) for k in WHOLE_FIELDS if k in r})
        out.append(r)
    return out


def year_gains(rec) -> dict:
    """{2022: gain, ...} from a scoreboard record (keys gain_2022 ... gain_2025)."""
    out = {}
    for y in YEARS:
        g = _num(rec.get(f"gain_{y}"))
        if g is not None:
            out[y] = g
    return out


def _years_text(gains: dict) -> str:
    if not gains:
        return "no yearly results"
    up = [str(y) for y, g in gains.items() if g > 0]
    down = [str(y) for y, g in gains.items() if g <= 0]
    if not down:
        return "helped in all 4 years"
    if not up:
        return "did not help in any year"
    worst = min(gains, key=gains.get)
    return f"helped in {', '.join(up)} but not in {', '.join(down)} (worst {worst}: {gains[worst]:+.3f})"


def _models_text(lr, rf) -> str:
    if lr is None or rf is None:
        return ""
    if lr > 0 and rf > 0:
        return "helped both models"
    if lr > 0:
        return "helped Logistic Regression only"
    if rf > 0:
        return "helped Random Forest only"
    return "helped neither model"


def why_line(rec, min_gain: float = 0.01) -> str:
    """One plain sentence explaining the result, written only from the measured numbers."""
    if rec.get("status") == "invalid":
        return f"could not be used: {rec.get('reason', '')}"
    parts = []
    gains = year_gains(rec)
    parts.append(_years_text(gains))
    m = _models_text(_num(rec.get("auc_gain_lr")), _num(rec.get("auc_gain_rf")))
    if m:
        parts.append(m)
    avg = _num(rec.get("auc_gain"))
    if avg is not None and 0 < avg < min_gain:
        parts.append(f"average gain {avg:+.3f} is too small to tell apart from luck")
    corr, closest = _num(rec.get("max_corr")), rec.get("closest", "")
    if corr is not None:
        if corr >= 0.9:
            parts.append(f"near-copy of {closest} ({corr:.2f})")
        elif corr >= 0.7:
            parts.append(f"quite similar to {closest} ({corr:.2f}), so little new information")
        elif corr < 0.3:
            parts.append("new information (not similar to any existing factor)")
    team = _num(rec.get("team_gain"))
    if team is not None and avg is not None and rec.get("team_size", 0):
        if avg > 0 >= team:
            parts.append("helped alone but added nothing on top of the selected team")
        elif team > 0 >= avg:
            parts.append("weak alone but helped on top of the selected team")
    if rec.get("best_pair_partner"):
        pg = _num(rec.get("best_pair_gain"))
        syn = _num(rec.get("best_pair_synergy"))
        if pg is not None and syn is not None and syn > 0:
            parts.append(f"worked better together with {rec['best_pair_partner']} "
                         f"(pair {pg:+.3f}, {syn:+.3f} more than the two alone added up)")
    return "; ".join(parts)


def report_card(rec, min_gain: float = 0.01) -> str:
    """A few lines about one factor, as shown to Claude."""
    status = str(rec.get("status", "")).upper()
    role = rec.get("role") or "-"
    mode = str(rec.get("mode")) + (f" (parent: {rec['parent']})" if rec.get("parent") else "")
    head = (f"[R{rec.get('round')}] {rec.get('name')} | {mode} | role: {role} | {status}"
            + (f" ({rec['reason']})" if rec.get("reason") else ""))
    lines = [head, f"  formula: {rec.get('formula')}"]
    if status != "INVALID":
        gains = year_gains(rec)
        years = ", ".join(f"{y} {g:+.3f}" for y, g in gains.items())
        avg, lr, rf = (_num(rec.get(k)) for k in ("auc_gain", "auc_gain_lr", "auc_gain_rf"))
        corr = _num(rec.get("max_corr"))
        perf = f"  alone (added to the 22): {years}"
        if avg is not None:
            perf += f" | avg {avg:+.3f}"
        if lr is not None and rf is not None:
            perf += f" | LR {lr:+.3f}, RF {rf:+.3f}"
        if corr is not None:
            perf += f" | similarity {corr:.2f} ({rec.get('closest', '')})"
        lines.append(perf)
        team = _num(rec.get("team_gain"))
        if team is not None and rec.get("team_size", 0):
            size = rec["team_size"]
            lines.append(f"  on top of the selected team ({size} factor{'' if size == 1 else 's'}): avg {team:+.3f}, "
                         f"helped {rec.get('team_years_helped', 0)}/4 years")
        if rec.get("best_pair_partner"):
            syn = _num(rec.get("best_pair_synergy"))
            lines.append(f"  best pair: with {rec['best_pair_partner']}: avg {_num(rec.get('best_pair_gain')):+.3f}, "
                         f"helped {rec.get('best_pair_years', 0)}/4 years"
                         + (f", {syn:+.3f} compared with the two alone added up" if syn is not None else ""))
    lines.append(f"  why: {why_line(rec, min_gain)}")
    if rec.get("lesson"):
        lines.append(f"  your lesson: {rec['lesson']}")
    return "\n".join(lines)


def one_liner(rec) -> str:
    avg = _num(rec.get("auc_gain"))
    avg_txt = f"{avg:+.3f}" if avg is not None else "n/a"
    return (f"[R{rec.get('round')}] {rec.get('name')} | {str(rec.get('status', '')).upper()} | avg {avg_txt} | "
            f"{why_line(rec)} | {rec.get('formula')}")


def lab_notebook(board: list, full_cards: int = 24) -> str:
    """Everything tried so far. Full report cards for the most useful entries, one line for the rest.

    Order: kept factors, then promising ones, then the latest round, then the best of the rest.
    """
    if not board:
        return "(empty: this is the first round)"
    last_round = max(int(r.get("round", 0)) for r in board)
    rank = {"kept": 0, "promising": 1}

    def key(r):
        avg = _num(r.get("auc_gain"))
        return (rank.get(r.get("status"), 2), 0 if int(r.get("round", 0)) == last_round else 1,
                -(avg if avg is not None else -9))

    ordered = sorted(board, key=key)
    full = ordered[:full_cards]
    rest = ordered[full_cards:]
    text = "\n\n".join(report_card(r) for r in full)
    if rest:
        text += "\n\nOLDER OR WEAKER ATTEMPTS (one line each):\n" + "\n".join(one_liner(r) for r in rest)
    return text

In [ ]:
# Section 3b: Report cards from the new run (C): a team member, a promising one and a rejected one
c_records = board_records(pd.read_csv(RUN_C / 'scoreboard.csv'))
examples = []
for status in ['kept', 'promising', 'rejected']:
    with_lesson = [r for r in c_records if r['status'] == status and r['lesson']]
    found = with_lesson or [r for r in c_records if r['status'] == status]
    if found:
        examples.append(found[0])
for r in examples:
    print(report_card(r), end='\n\n')

**Reading this output**

* Each card has everything Dr Liu listed: the formula, the gain in each year, Logistic Regression vs Random Forest, kept/promising/rejected with the reason, the similarity to the closest existing factor, an automatic "why" line written only from the numbers, and **Claude's own lesson**, written in the next round.
* Run C selected no factor, so the cards shown are a promising one and a rejected one.

## Step 4: What I Tell Claude Now

The instructions have the same parts as last week (goal, data, the 22 factors, formula language), plus three new parts: **the selected team**, **the promising list** and **the pair trials so far**. The scoreboard is replaced by the report cards. The task now asks for improve, new and partner factors, and a lesson for every factor of the last round.

In [ ]:
# Section 4a: The new instructions for Claude  (also saved as src/stage2/prompt_v2.py)
#
# Instructions for Claude in the interaction-aware loop (version 2, after Liu's 20 Oct feedback).
#
# Differences from version 1 (llm.build_prompt):
#   - Claude sees a full report card for every factor tried (feedback.lab_notebook):
#     yearly results, which model it helped, similarity, why it worked or failed, its own lesson.
#   - Claude sees the selected team, the promising list and the pair results.
#   - Each round asks for IMPROVE (fix a factor that worked only partly), NEW and PARTNER
#     (a factor designed to work together with a promising or selected one) factors.
#   - Claude labels each factor's role, so complementary pairs can be found
#     (for example a market-condition factor with an Apple-relative factor).
#   - Claude writes a one-line lesson for every factor from the previous round.

ROLES = {
    "market_condition": "the state of the whole market or tech sector (e.g. QQQ trend or volatility, how much the peers disagree)",
    "relative_move": "Apple compared with the market or with MSFT/NVDA",
    "apple_own": "Apple's own price, range or volume pattern",
    "peer_lead": "moves of QQQ, MSFT or NVDA that might lead Apple's next move",
}

TOOL_V2 = {
    "name": "propose_factors",
    "description": "Return your lessons from the last round and the proposed factors.",
    "input_schema": {
        "type": "object",
        "properties": {
            "reflection": {"type": "string",
                           "description": "3-5 sentences: what the lab notebook shows works or fails, in which years and for which model, and what you will try."},
            "lessons": {
                "type": "array",
                "description": "One entry for every factor from the previous round (empty in round 1).",
                "items": {"type": "object",
                          "properties": {"name": {"type": "string"},
                                         "lesson": {"type": "string", "description": "one sentence: why it probably worked or failed"}},
                          "required": ["name", "lesson"]},
            },
            "factors": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string", "description": "snake_case, unique, descriptive"},
                        "mode": {"type": "string", "enum": ["improve", "new", "partner"]},
                        "parent": {"type": "string", "description": "improve: the factor being fixed; partner: the factor it should work with; new: empty"},
                        "role": {"type": "string", "enum": list(ROLES)},
                        "formula": {"type": "string"},
                        "hypothesis": {"type": "string", "description": "one or two sentences: why it could help, and for improve/partner what you changed or how it complements the parent"},
                    },
                    "required": ["name", "mode", "parent", "role", "formula", "hypothesis"],
                },
            },
        },
        "required": ["reflection", "lessons", "factors"],
    },
}

SYSTEM_V2 = """You are a quantitative researcher searching for interpretable daily factors, working like a scientist with a lab notebook.
Every factor you propose is tested automatically, alone and in combination with others, and the full results come back to you.
Learn from them: work out why a factor helped in some years or for one model and not others, then fix it;
design factors that work together; drop ideas that keep failing. Never use future information.
Write formulas only with the operators and inputs listed."""


def rules_text(rules) -> str:
    if len(rules.get("checks", ["q3"])) > 1:      # 8 checks: Q2 (trained on Q1) and Q3 (trained on Q1-Q2) each year
        text = rules_text({**rules, "checks": ["q3"]}).replace("of 4 years", "of 8 checks")
        return text + ("\nEach factor is checked twice per year: on Q2 after training on Q1, and on Q3 after training "
                       "on Q1-Q2, so there are 8 checks. Yearly results shown are the Q3 checks.")
    return (f"A factor joins the SELECTED TEAM if, on top of the 22 factors and the current team, it helps in at least "
            f"{rules['min_years']} of 4 years, the average gain is at least {rules['team_gain']:.3f}, and its similarity "
            f"to every existing and selected factor is below {rules['max_corr']:.2f}. The team holds at most {rules['team_size']} factors.\n"
            f"A PAIR joins the team together if the two factors added together help in at least {rules['min_years']} of 4 years, "
            f"the pair's average gain is at least {rules['pair_gain']:.3f}, and the pair's gain is at least "
            f"{rules['pair_synergy']:.3f} more than the two factors' gains (each on top of the team) added up. Two factors "
            f"that each got lucky simply add up their luck; real teamwork gives more than the sum.\n"
            f"A factor that is weak alone but shows some sign of life (helped in 2+ years, or helped one model) stays on the "
            f"PROMISING list and can still join the team later in a pair. Nothing is rejected only for being weak alone.\n"
            f"For reference, factors made of pure random numbers rarely pass these bars; small gains are noise."
            + ("\nGRADED ON A CURVE: every gain shown for a factor alone or on the team is its gain MINUS the typical "
               "(median) gain of a fixed class of 24 template formulas (no LLM) in the same year and on the same team. "
               "Each year's validation quarter reacts to almost any new factor in the same way (in 2024 nearly every "
               "factor lowered the score, in 2023 nearly every factor raised it), so positive now means 'better than a "
               "typical factor in that year'. Pair trials are not curved." if rules.get("curve") else ""))


def build_prompt_v2(*, round_no, n_improve, n_new, n_partner, base_formulas, stage1_top, market,
                    notebook, team_text, promising_text, pairs_text, rules, previous_names):
    base = "\n".join(f"- {k} = {v}" for k, v in base_formulas.items())
    roles = "\n".join(f"- {k}: {v}" for k, v in ROLES.items())
    lessons_task = ("Write a one-sentence lesson for each of these factors from the previous round: "
                    + ", ".join(previous_names)) if previous_names else "Round 1: no lessons yet (return an empty list)."
    return f"""ROUND {round_no}

GOAL
Predict whether AAPL closes higher tomorrow than today (binary). A factor is computed after the close of day t.

DATA (daily, adjusted OHLCV, Sept 2021 - Dec 2025)
Inputs: {", ".join(INPUTS)}
Market summary up to 30 June 2025:
{market}

HOW EACH FACTOR IS TESTED
For each year 2022, 2023, 2024 and 2025: train on Q1-Q2, score on Q3 (validation). Q4 is never used in this search.
The score is ROC AUC; "gain" is the change in AUC when a factor is added, averaged over Logistic Regression (LR)
and Random Forest (RF), and shown for each year and each model.
1. Alone: the factor is added to the 22 existing factors.
2. Team trial: the factor is added to the 22 factors plus the selected team.
3. Pair trial: promising factors of different roles are added two at a time.
{rules_text(rules)}

THE 22 EXISTING FACTORS (do not repeat them)
{base}

MOST USEFUL EXISTING FACTORS IN STAGE 1
{stage1_top}

FACTOR ROLES (label every factor with one)
{roles}

THE SELECTED TEAM
{team_text}

PROMISING LIST (weak alone, may work in a pair; good parents for PARTNER factors)
{promising_text}

PAIR TRIALS SO FAR
{pairs_text}

YOUR LAB NOTEBOOK (every factor tried so far)
{notebook}

FORMULA LANGUAGE
Nested function calls only, no + - * / symbols. Windows n must be one of {WINDOWS}.
{operator_guide()}
Example: safe_divide(rolling_std(pct_change(AAPL_Close, 1), 5), rolling_std(pct_change(QQQ_Close, 1), 20))

TASK
1. {lessons_task}
2. Propose {n_improve} IMPROVE factors: pick factors from the notebook that helped in some years or for one model,
   say in the hypothesis why they probably failed where they failed, and change them to fix that. Parent = that factor.
3. Propose {n_new} NEW factors: genuinely different ideas from everything in the notebook. Parent is empty.
4. Propose {n_partner} PARTNER factors: each designed to work together with one factor on the selected team or the
   promising list (parent = that factor), with a different role. For example, if the parent describes the market
   condition, design one that describes Apple's relative move, and the other way round.
Label every factor with its role. Do not resubmit any formula from the notebook.
Answer ONLY by calling the tool propose_factors (no other text)."""

In [ ]:
# Section 4b: The new parts of the instructions Claude got in round 4 of run C
prompt = (RUN_C / 'round_04_prompt.txt').read_text()
start, end = prompt.index('THE SELECTED TEAM'), prompt.index('YOUR LAB NOTEBOOK')
print(prompt[start:end])
print(prompt[prompt.index('TASK'):])

In [ ]:
# Section 4c: What Claude wrote back in round 4: its reflection, its lessons, and the factors it proposed
reply = json.loads((RUN_C / 'round_04_response.json').read_text())
print('REFLECTION:', reply['reflection'], '\n')
print('LESSONS ABOUT ROUND 3:')
for item in reply.get('lessons', []):
    print(f"- {item['name']}: {item['lesson']}")
print('\nFACTORS PROPOSED:')
for f in reply['factors']:
    parent = f" (parent: {f['parent']})" if f.get('parent') else ''
    print(f"- [{f['mode']}{parent}] {f['name']} | role: {f['role']}\n    {f['formula']}\n    {f['hypothesis']}")

**Reading this output**

* Claude used the feedback. Its lessons are specific and refer to years and models, for example that smoothing volume over several days "removed the one-day attention spike that carried the signal".
* From round 2 on, Claude itself pointed out that almost every factor helped in 2023 and 2025 and hurt in 2024. Step 9 shows this is true for every kind of factor.
* Each round it asked for 2 improve, 2 new and 2 partner factors, and it labelled each factor's role (market condition, Apple-relative move, Apple's own pattern, peer lead).

## Step 5: Team Trial and Pair Trial

The scoring code from last week now takes two inputs: the factor(s) to add, and the team they are added on top of.
* **Alone:** add one factor to my 22 factors (exactly last week's test; the numbers are identical)
* **Team trial:** add one factor to my 22 factors plus the selected team
* **Pair trial:** add two factors at once to my 22 factors plus the selected team

The gain is always "check score (ROC AUC on Q3) with the new factor(s) minus without", for each of the four years, averaged over Logistic Regression and the three Random Forests.

In [ ]:
# Section 5a: Testing factors alone, on top of the team, or in a pair  (also saved as src/stage2/scoring.py)
#
# Score candidate factors against the 22 Stage 1 factors (validation quarters only).
#
# For every year the same quarterly split as Stage 1 is used (Q1-Q2 train,
# Q3 validation, Q4 test). A candidate is added to feature set C and the
# Stage 1 models are refitted on the training quarters. Only the validation
# quarter is scored, so the Q4 test quarters stay untouched for the final test.
#
# Reported per candidate and year:
#   auc_gain      validation ROC AUC with the candidate minus without it
#                 (mean of Logistic Regression and the 3 Random Forest seeds)
#   bal_acc_gain  same for balanced accuracy
#   importance    permutation importance of the candidate (drop in validation AUC)
#   helped        auc_gain > 0

from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

from src.data import quarterly_split
from src.features import FEATURE_SETS
from src.models import fit_logistic_regression, fit_random_forest

TARGET = "target_next_day_up"
BASE = FEATURE_SETS["C"]
YEARS = (2022, 2023, 2024, 2025)
LR_SEED, RF_SEEDS = 42, (0, 1, 2)
# Inputs the forest considers at each split. Stage 1's default ("sqrt") gives 4 for 22-24 factors but 5 from
# 25 factors on, so adding a 3rd team factor (or a pair on top of a 1-factor team) would change the forest
# itself, and even random numbers then looked useful (20 Oct check). Fixed at 4, the same as Stage 1 uses
# for the 22 factors, so the 22-factor models and every single-factor test are unchanged.
RF_MAX_FEATURES = 4


def _models(x, y):
    yield "logistic_regression", fit_logistic_regression(x, y, LR_SEED)
    for seed in RF_SEEDS:
        yield f"random_forest_{seed}", fit_random_forest(x, y, seed, max_features=RF_MAX_FEATURES)


def _fit_score(train, val, columns, importance_of=None, repeats=20):
    rows = []
    for name, model in _models(train[columns], train[TARGET]):
        proba = model.predict_proba(val[columns])[:, 1]
        row = {"model": name,
               "auc": roc_auc_score(val[TARGET], proba),
               "bal_acc": balanced_accuracy_score(val[TARGET], (proba >= 0.5).astype(int))}
        if importance_of is not None:
            # permutation importance of this one column only (same idea as sklearn's, much faster)
            rng = np.random.default_rng(0)
            x = val[columns].copy()
            drops = []
            for _ in range(repeats):
                x[importance_of] = rng.permutation(val[importance_of].to_numpy())
                drops.append(row["auc"] - roc_auc_score(val[TARGET], model.predict_proba(x)[:, 1]))
            row["importance"] = float(np.mean(drops))
        rows.append(row)
    return pd.DataFrame(rows)


def _check_split(data, year, check):
    """Training and checking days for one check.
    'q3': train on Q1-Q2, check on Q3 (the Stage 1 validation quarter).
    'q2': train on Q1, check on Q2 (an extra, earlier check; Q1 days whose next day is in Q2 are dropped)."""
    s = quarterly_split(data, year)
    if check == "q3":
        return s["train"], s["validation"]
    train = s["train"]
    q1 = train[(train["Date"].dt.quarter == 1) & (pd.to_datetime(train["target_date"]).dt.quarter == 1)]
    return q1.reset_index(drop=True), train[train["Date"].dt.quarter == 2].reset_index(drop=True)


@dataclass
class Scorer:
    """Caches the base-model scores so each new candidate costs one refit per year and check."""
    features: pd.DataFrame          # Stage 1 feature table (build_stage1_features output)
    years: tuple = YEARS
    checks: tuple = ("q3",)         # ("q2", "q3") = 8 checks instead of 4 (longer measuring stick, 21 Oct)

    def __post_init__(self):
        self.features = self.features.copy()
        self.features["Date"] = pd.to_datetime(self.features["Date"])
        self._base = {}          # (year, check, team names) -> scores of the models without the new factor(s)

    def score(self, name: str, values: pd.Series) -> pd.DataFrame:
        """values: factor indexed by Date. Returns one row per year (gain over the 22 factors)."""
        return self.gains({name: values}, team={}, importance_of=name)

    def gains(self, add: dict, team: dict, importance_of=None) -> pd.DataFrame:
        """Gain from adding the factor(s) in `add` on top of the 22 factors plus `team`.

        add, team: {name: values indexed by Date}. With an empty team this is the solo test;
        with the selected team it is the team trial; with two factors in `add` it is a pair trial.
        Returns one row per year: auc_gain (mean of LR and the 3 RF seeds), auc_gain_lr,
        auc_gain_rf, bal_acc_gain and, if requested, the permutation importance of one factor.
        """
        add_names, team_names = list(add), sorted(team)
        label = "+".join(add_names)
        data = self.features
        for name, values in {**team, **add}.items():
            values = pd.Series(values, name=name)
            values.index = pd.to_datetime(values.index)
            data = data.merge(values.rename(name), left_on="Date", right_index=True, how="left")
        extra = team_names + add_names
        rows = []
        for year in self.years:
            for check in self.checks:
                train, val = _check_split(data, year, check)
                bad = [c for c in extra if train[c].isna().any() or val[c].isna().any()
                       or not np.isfinite(train[c]).all() or train[c].nunique() < 3]
                if bad:
                    rows.append({"factor": label, "year": year, "check": check, "valid": False})
                    continue
                # tame extreme values using the training quarters' 1st/99th percentiles (no peeking at validation)
                train, val = train.copy(), val.copy()
                for c in extra:
                    lo, hi = train[c].quantile([0.01, 0.99])
                    train[c], val[c] = train[c].clip(lo, hi), val[c].clip(lo, hi)
                key = (year, check, tuple(team_names))
                if key not in self._base:
                    self._base[key] = _fit_score(train, val, BASE + team_names).set_index("model")
                base = self._base[key]
                cand = _fit_score(train, val, BASE + team_names + add_names, importance_of=importance_of).set_index("model")
                gain = cand[["auc", "bal_acc"]] - base[["auc", "bal_acc"]]
                row = {"factor": label, "year": year, "check": check, "valid": True,
                       "auc_gain": gain["auc"].mean(),
                       "auc_gain_lr": gain.loc["logistic_regression", "auc"],
                       "auc_gain_rf": gain.drop("logistic_regression")["auc"].mean(),
                       "bal_acc_gain": gain["bal_acc"].mean()}
                row["importance"] = cand["importance"].mean() if importance_of is not None else float("nan")
                rows.append(row)
        return pd.DataFrame(rows)


def summarise(per_year: pd.DataFrame) -> pd.DataFrame:
    """One row per factor: averages over years and how many years it helped."""
    ok = per_year[per_year["valid"].astype(bool)]
    out = ok.groupby("factor").agg(
        auc_gain=("auc_gain", "mean"),
        auc_gain_lr=("auc_gain_lr", "mean"),
        auc_gain_rf=("auc_gain_rf", "mean"),
        bal_acc_gain=("bal_acc_gain", "mean"),
        importance=("importance", "mean"),
        years_helped=("auc_gain", lambda x: int((x > 0).sum())),
        years_scored=("auc_gain", "size"),
    )
    return out.sort_values(["years_helped", "auc_gain"], ascending=False)

In [ ]:
# Section 5b: The rules for the team, the pairs and the promising list  (also saved as src/stage2/selection.py)
#
# Keep rules and helpers for the interaction-aware search (team trial, pair trials, promising list).
#
# Plain-language summary:
#   - Solo test: the factor is added to the 22 Stage 1 factors (as in version 1).
#   - Team trial: the factor is added to the 22 factors plus the selected team.
#     It joins the team if the team gets clearly better, consistently across years.
#   - Pair trial: two promising factors, preferably of different kinds, are added together.
#     Both join the team if together they do clearly better than their two gains added up, consistently
#     (two lucky factors only add up their luck; real teamwork gives more than the sum).
#   - Promising list: factors that are weak alone but show some sign of life are not thrown away.

from dataclasses import dataclass, asdict

import pandas as pd

YEARS = (2022, 2023, 2024, 2025)


@dataclass
class Rules:
    min_years: int = 3            # must help in at least 3 of the 4 years
    team_gain: float = 0.01       # team trial: average gain needed (set from the random-number check)
    pair_gain: float = 0.015      # pair trial: average gain of the pair needed
    pair_synergy: float = 0.005   # pair gain must exceed the two members' gains added up by this much
    max_corr: float = 0.9         # anything this similar to an existing or selected factor is a copy
    pair_max_corr: float = 0.7    # the two members of a pair should be different from each other
    team_size: int = 6            # the selected team stays small
    pairs_per_round: int = 6      # pair trials per round (keeps the number of "lottery tickets" low)
    curve: bool = False           # grade alone/team results on a curve (src/stage2/curve.py); pairs stay raw
    checks: tuple = ("q3",)       # ("q2", "q3"): check on Q2 (trained on Q1) as well as Q3 -> 8 checks; min_years counts checks

    def as_dict(self):
        return asdict(self)


def summarise_result(per_year: pd.DataFrame) -> dict:
    """Turn the per-year table from Scorer.gains into a few numbers."""
    if per_year.empty or not per_year["valid"].astype(bool).all():
        return {"valid": False}
    g = per_year if "check" in per_year else per_year.assign(check="q3")
    q3 = g[g["check"] == "q3"].set_index("year")
    return {"valid": True,
            "gain": float(g["auc_gain"].mean()),
            "years": int((g["auc_gain"] > 0).sum()),          # checks helped (4 checks = years; 8 with Q2 too)
            "n_checks": int(len(g)),
            "lr": float(g["auc_gain_lr"].mean()),
            "rf": float(g["auc_gain_rf"].mean()),
            "by_year": {int(y): float(v) for y, v in q3["auc_gain"].items()},
            "by_check": {f"{int(y)}{c}": float(v) for y, c, v in zip(g["year"], g["check"], g["auc_gain"])}}


def group_of(formula: str, role: str | None = None) -> str:
    """Which kind of clue it is: 'market' (market or peers), 'relative' (Apple vs others) or 'apple' (Apple only).

    Uses Claude's role label when there is one, otherwise the stocks the formula uses.
    """
    if role in ("market_condition", "peer_lead"):
        return "market"
    if role == "relative_move":
        return "relative"
    if role == "apple_own":
        return "apple"
    uses_aapl = "AAPL_" in formula
    uses_other = any(f"{t}_" in formula for t in ("QQQ", "MSFT", "NVDA"))
    if uses_aapl and uses_other:
        return "relative"
    return "apple" if uses_aapl else "market"


def passes(result: dict, rules: Rules, gain_bar: float) -> bool:
    return bool(result.get("valid")) and result["years"] >= rules.min_years and result["gain"] >= gain_bar


def is_promising(rec: dict, rules: Rules) -> bool:
    """Weak alone but some sign of life: positive average and helped in 2+ years, alone or on the team, and not a copy."""
    if rec.get("status") == "invalid" or rec.get("max_corr", 1) >= rules.max_corr:
        return False
    best_gain = max(rec.get("auc_gain", -1), rec.get("team_gain", -1) if rec.get("team_size") else -1)
    best_years = max(rec.get("years_helped", 0), rec.get("team_years_helped", 0) if rec.get("team_size") else 0)
    return best_gain > 0 and best_years >= 2 * len(rules.checks)     # helped in at least half the checks


def choose_pairs(new_names, pool_names, info, corr, rules: Rules, forced=()):
    """Pick which pairs to try this round.

    forced: (partner, parent) pairs Claude designed to work together; tried first.
    Then each new promising factor is paired with factors on the promising list, preferring
    different kinds (e.g. a market-condition factor with an Apple-relative one) and the
    strongest signs of life. info[name] = {"group", "score"}; corr(a, b) = similarity of two factors.
    """
    chosen, seen = [], set()

    def add(a, b):
        key = tuple(sorted((a, b)))
        if a == b or key in seen or len(chosen) >= rules.pairs_per_round:
            return
        if abs(corr(a, b)) >= rules.pair_max_corr:
            return
        seen.add(key)
        chosen.append((a, b))

    for a, b in forced:
        if a in info and b in info:
            add(a, b)
    candidates = []
    for a in new_names:
        for b in pool_names:
            if a == b or a not in info or b not in info:
                continue
            bonus = 0.005 if info[a]["group"] != info[b]["group"] else 0.0
            candidates.append((info[a]["score"] + info[b]["score"] + bonus, a, b))
    for _, a, b in sorted(candidates, reverse=True):
        add(a, b)
    return chosen

The loop itself. One round: ask Claude, check and compute each formula, test each factor alone, run the team trial, run a few pair trials, then update the team, the promising list and the report cards.

In [ ]:
# Section 5c: Grading on a curve (tried in Step 10, not used)  (also saved as src/stage2/curve.py)
#
# Grading on a curve: compare a factor with what a typical factor does in the same year.
#
# Why (21 Oct): in each year the validation quarter (Q3) reacts to ANY new price-based factor in a similar way.
# In 2024 about 90% of all factors made the score worse (Claude's, last week's and the template formulas with no
# LLM alike); in 2023 about 80-90% made it better. So "helped in 3 of 4 years" mostly measured the years, not
# the factors. Like a hard exam that almost everyone failed, the fair question is: did this factor do better
# than a typical factor did in that year?
#
# A fixed "class" of template formulas (no LLM, made by RandomProposer with a fixed seed) is tested the same
# way as each factor, on top of the same team. A factor's curved gain in a year = its gain minus the median
# gain of the class in that year (for the average, for Logistic Regression and for Random Forest separately).
# The class is chosen before any real run and never changes.

import pandas as pd


PANEL_SIZE, PANEL_SEED = 24, 101


def template_formulas(n, seed, prices, exclude=()):
    """n different valid template formulas (no LLM) with their values: {name: (formula, values)}."""
    rng, out, seen, tries = RandomProposer(seed), {}, set(exclude), 0
    while len(out) < n and tries < 50 * n:
        tries += 1
        f = rng._formula()
        if f in seen:
            continue
        seen.add(f)
        try:
            validate(f)
            v = evaluate(f, prices)
        except (FormulaError, KeyError, ValueError):
            continue
        if v.loc["2022":].isna().any() or v.loc["2022":].nunique() < 10:
            continue
        out[f"template_{seed}_{len(out):02d}"] = (f, v)
    return out


class Curve:
    """Median gain of the template class in each year, on top of a given team (cached per team)."""

    def __init__(self, scorer, prices, size=PANEL_SIZE, seed=PANEL_SEED):
        self.scorer = scorer
        self.panel = template_formulas(size, seed, prices)
        self._offsets = {}

    def offsets(self, team: dict) -> pd.DataFrame:
        key = tuple(sorted(team))
        if key not in self._offsets:
            rows = [self.scorer.gains({n: v}, team) for n, (_, v) in self.panel.items()]
            ok = [r for r in rows if r["valid"].astype(bool).all()]
            all_rows = pd.concat(ok, ignore_index=True)
            self._offsets[key] = all_rows.groupby("year")[["auc_gain", "auc_gain_lr", "auc_gain_rf"]].median()
        return self._offsets[key]

    def apply(self, per_year: pd.DataFrame, team: dict) -> pd.DataFrame:
        """The same table with every gain replaced by its curved gain (gain minus the class median)."""
        if per_year.empty or not per_year["valid"].astype(bool).all():
            return per_year
        off = self.offsets(team)
        out = per_year.copy()
        for c in ("auc_gain", "auc_gain_lr", "auc_gain_rf"):
            out[c] = out[c] - out["year"].map(off[c])
        return out

In [ ]:
# Section 5d: The new loop  (also saved as src/stage2/loop_v2.py)
#
# Interaction-aware iterative factor search (version 2, after Liu's 20 Oct feedback).
#
# Every round:
#   1. Claude gets the full lab notebook (report card for every factor tried, the selected team,
#      the promising list and the pair results) and returns lessons for the last round plus
#      IMPROVE, NEW and PARTNER factors.
#   2. Each formula is checked and calculated (past data only).
#   3. Solo test: each factor added to the 22 Stage 1 factors (same as version 1).
#   4. Team trial: each factor added to the 22 factors plus the selected team; the best ones that
#      clearly improve the team join it (best first, team stays small).
#   5. Pair trials: a few promising factors, preferably of different kinds, are tested two at a time;
#      a pair that does clearly better than its two members' gains added up joins the team together
#      (two lucky factors only add up their luck; real teamwork gives more than the sum).
#   6. Factors that are weak alone but show some sign of life stay on the promising list.
#   7. Everything is saved and goes back to Claude in the next round.
# finalize() gives the promising list one more team trial at the end. It is off by default in
# experiments/run_stage2_loop_v2.py because the random-number check showed it lets luck in (1 in 18).
# The Q4 test quarters are never used here.

import json
import re
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd



def best_gain(rec):
    """Best average gain seen for a factor: alone, or on top of the selected team."""
    team = rec.get("team_gain", -1) if rec.get("team_size") else -1
    return max(rec.get("auc_gain", -1), team)


@dataclass
class LoopV2:
    proposer: object
    features: pd.DataFrame
    prices: pd.DataFrame
    out_dir: Path
    stage1_importance_path: Path
    rules: Rules = field(default_factory=Rules)
    n_improve: int = 2
    n_new: int = 2
    n_partner: int = 2
    board: list = field(default_factory=list)     # one record per factor tried
    pairs: list = field(default_factory=list)     # one record per pair trial
    team: list = field(default_factory=list)      # names of the selected team, in the order they joined
    log: list = field(default_factory=list)       # one summary per round

    def __post_init__(self):
        self.out_dir = Path(self.out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.scorer = Scorer(self.features, checks=tuple(self.rules.checks))
        self.market = market_summary(self.prices)
        self.stage1_top = stage1_importance(self.stage1_importance_path)
        dates = pd.to_datetime(self.features["Date"])
        self.check_idx = pd.DatetimeIndex(dates[(dates.dt.year >= 2022) & (dates.dt.month <= 9)])   # Q1-Q3 only
        self.reference = self.features.set_index(pd.to_datetime(self.features["Date"]))[BASE]
        self.values, self.solo = {}, {}
        self._py = {}            # (name, team) -> raw per-year table from the scorer
        self.curve = Curve(self.scorer, self.prices) if self.rules.curve else None
        self._team_cache, self._corr_cache = {}, {}

    # ---------------- small helpers ----------------
    def rec(self, name):
        return next(r for r in self.board if r["name"] == name)

    def team_values(self):
        return {n: self.values[n] for n in self.team}

    def similarity(self, name):
        """Largest |Spearman correlation| with the 22 factors and the selected team (Q1-Q3 days)."""
        ref = self.reference.copy()
        for n in self.team:
            if n != name:
                ref[n] = self.values[n].reindex(ref.index)
        x = self.values[name].reindex(ref.index).loc[self.check_idx]
        c = ref.loc[self.check_idx].corrwith(x, method="spearman").abs()
        return float(c.max()), str(c.idxmax())

    def corr(self, a, b):
        key = tuple(sorted((a, b)))
        if key not in self._corr_cache:
            xa, xb = self.values[a].reindex(self.check_idx), self.values[b].reindex(self.check_idx)
            self._corr_cache[key] = float(xa.corr(xb, method="spearman"))
        return self._corr_cache[key]

    def per_year(self, name, team_names=None):
        """Raw per-year table for the factor on top of the 22 factors plus a team (default: current team)."""
        team_names = tuple(sorted(self.team if team_names is None else team_names))
        key = (name, team_names)
        if key not in self._py:
            self._py[key] = self.scorer.gains({name: self.values[name]}, {n: self.values[n] for n in team_names})
        return self._py[key]

    def judged(self, per_year, team_names):
        """Summary used for decisions: graded on the curve when the rules say so."""
        if self.curve is None:
            return summarise_result(per_year)
        return summarise_result(self.curve.apply(per_year, {n: self.values[n] for n in team_names}))

    def team_result(self, name, raw=False):
        """The factor added on top of the 22 factors plus the current team (curved unless raw=True)."""
        key = (name, tuple(sorted(self.team)), raw)
        if key not in self._team_cache:
            py = self.per_year(name)
            self._team_cache[key] = summarise_result(py) if raw else self.judged(py, self.team)
        return self._team_cache[key]

    # ---------------- one factor ----------------
    def _build(self, cand, round_no):
        rec = {"round": round_no, "name": str(cand.get("name", "?")), "mode": cand.get("mode", "?"),
               "parent": cand.get("parent", "") or "", "role": cand.get("role", "") or "",
               "formula": str(cand.get("formula", "")), "hypothesis": cand.get("hypothesis", ""),
               "status": "invalid", "reason": "", "lesson": "", "how_selected": ""}
        names = {r["name"] for r in self.board} | set(BASE)
        base_name, k = rec["name"], 1
        while rec["name"] in names:
            rec["name"] = f"{base_name}_r{round_no}" + (f"_{k}" if k > 1 else "")
            k += 1
        formulas = {r["formula"].replace(" ", "") for r in self.board} | \
                   {f.replace(" ", "") for f in BASE_FORMULAS.values()}
        if rec["formula"].replace(" ", "") in formulas:
            rec["reason"] = "duplicate formula"
            return rec
        try:
            validate(rec["formula"])
            values = evaluate(rec["formula"], self.prices)
        except (FormulaError, KeyError, ValueError) as exc:
            rec["reason"] = str(exc)[:200]
            return rec
        if values.loc["2022":].isna().any():
            rec["reason"] = "missing values in 2022-2025 (e.g. division by zero)"
            return rec
        self.values[rec["name"]] = values
        rec["group"] = group_of(rec["formula"], rec["role"])
        rec["status"] = "tested"
        return rec

    def _write_solo(self, rec, res):
        rec.update(auc_gain=res["gain"], years_helped=res["years"], auc_gain_lr=res["lr"], auc_gain_rf=res["rf"],
                   **{f"gain_{y}": g for y, g in res["by_year"].items()})

    def _write_team(self, rec, res):
        rec.update(team_size=len(self.team), team_gain=res["gain"], team_years_helped=res["years"],
                   team_lr=res["lr"], team_rf=res["rf"])

    # ---------------- team trial ----------------
    def _grow_team(self, recs, how):
        """Team trial for each factor. The best one that clearly improves the team joins it, then the rest are
        re-tested on the bigger team, until none passes. When the team is full the trial is still run and
        reported, but nothing joins."""
        while True:
            best = None
            for r in recs:
                if r["status"] == "kept":
                    continue
                res = self.team_result(r["name"])
                if not res.get("valid"):
                    continue
                self._write_team(r, res)
                r["max_corr"], r["closest"] = self.similarity(r["name"])
                if passes(res, self.rules, self.rules.team_gain) and r["max_corr"] < self.rules.max_corr:
                    if best is None or res["gain"] > best[1]["gain"]:
                        best = (r, res)
            if best is None:
                return
            r, res = best
            if len(self.team) >= self.rules.team_size:
                r["team_full"] = True
                return
            self.team.append(r["name"])
            r["status"], r["how_selected"] = "kept", how
            r["reason"] = f"joined the team ({how}): gain on the team {res['gain']:+.3f}, helped {res['years']}/4 years"

    # ---------------- pair trials ----------------
    def _pair_trials(self, round_no, new_names):
        """A few pair trials. PARTNER factors are always tried with their parent (they were designed to work
        together), even when weak alone. Other pairs: this round's promising factors with the promising list."""
        pool = [r for r in self.board if r["status"] in ("tested", "promising") and r["name"] in self.solo
                and is_promising(r, self.rules)]
        info = {r["name"]: {"group": r.get("group", ""), "score": best_gain(r)} for r in pool}
        forced = []
        for n in new_names:
            r = self.rec(n)
            p = next((x for x in self.board if x["name"] == r["parent"]), None)
            if (r["mode"] == "partner" and p is not None and p["status"] in ("tested", "promising", "rejected")
                    and p["name"] in self.solo and r.get("max_corr", 1) < self.rules.max_corr
                    and p.get("max_corr", 1) < self.rules.max_corr):
                for x in (r, p):
                    info.setdefault(x["name"], {"group": x.get("group", ""), "score": best_gain(x)})
                forced.append((n, p["name"]))
        new_promising = [n for n in new_names if n in {r["name"] for r in pool}]
        pool_names = [r["name"] for r in pool]
        for a, b in choose_pairs(new_promising, pool_names, info, self.corr, self.rules, forced):
            if a in self.team or b in self.team:
                continue
            res = summarise_result(self.scorer.gains({a: self.values[a], b: self.values[b]}, self.team_values()))
            ra, rb = self.team_result(a, raw=True), self.team_result(b, raw=True)    # pairs are judged raw
            if not (res.get("valid") and ra.get("valid") and rb.get("valid")):
                continue
            if self.team:                                  # keep each member's team result up to date
                self._write_team(self.rec(a), self.team_result(a))
                self._write_team(self.rec(b), self.team_result(b))
            synergy = res["gain"] - (ra["gain"] + rb["gain"])    # extra over the two gains added up
            room = len(self.team) + 2 <= self.rules.team_size
            not_copies = all(self.similarity(x)[0] < self.rules.max_corr for x in (a, b))   # vs the current team
            joined = (passes(res, self.rules, self.rules.pair_gain) and synergy >= self.rules.pair_synergy
                      and room and not_copies)
            self.pairs.append({"round": round_no, "a": a, "b": b,
                               "group_a": info[a]["group"], "group_b": info[b]["group"],
                               "team_size": len(self.team), "gain": res["gain"], "years": res["years"],
                               "lr": res["lr"], "rf": res["rf"],
                               **{f"gain_{y}": g for y, g in res["by_year"].items()},
                               "a_alone_on_team": ra["gain"], "b_alone_on_team": rb["gain"],
                               "synergy": synergy, "joined": joined})
            for x, y in ((a, b), (b, a)):
                rx = self.rec(x)
                if rx.get("best_pair_gain") is None or res["gain"] > rx["best_pair_gain"]:
                    rx.update(best_pair_partner=y, best_pair_gain=res["gain"], best_pair_years=res["years"],
                              best_pair_synergy=synergy)
            if joined:
                for x, y in ((a, b), (b, a)):
                    rx = self.rec(x)
                    self.team.append(x)
                    rx["status"], rx["how_selected"] = "kept", f"pair trial with {y}, round {round_no}"
                    rx["reason"] = (f"joined the team as a pair with {y}: pair gain {res['gain']:+.3f}, helped "
                                    f"{res['years']}/4 years, {synergy:+.3f} more than the two alone added up")

    # ---------------- statuses ----------------
    def _set_status(self, r):
        if r["status"] in ("kept", "invalid"):
            return
        result = f"alone {r.get('auc_gain', float('nan')):+.3f} ({r.get('years_helped', 0)}/4 years)"
        if r.get("team_size"):
            result += f", on the team {r['team_gain']:+.3f} ({r['team_years_helped']}/4 years)"
        if r.pop("team_full", False):
            r["status"] = "promising"
            r["reason"] = f"passed the team trial but the team is already full ({self.rules.team_size}): {result}"
            return
        if is_promising(r, self.rules):
            r["status"] = "promising"
            r["reason"] = f"not enough for the team yet but some sign of life: {result}"
            return
        r["status"] = "rejected"
        if r.get("max_corr", 0) >= self.rules.max_corr:
            r["reason"] = f"near-copy of {r.get('closest', '')} ({r['max_corr']:.2f})"
        else:
            r["reason"] = f"too weak: {result}"

    # ---------------- text for Claude ----------------
    def team_text(self):
        if not self.team:
            return "(empty so far)"
        return "\n".join(f"- {n} [{self.rec(n).get('group', '')}] = {self.rec(n)['formula']} "
                         f"(joined: {self.rec(n)['how_selected']})" for n in self.team)

    def promising_text(self, limit=12):
        pool = [r for r in self.board if r["status"] == "promising"]
        pool.sort(key=lambda r: -best_gain(r))
        if not pool:
            return "(empty so far)"
        lines = []
        for r in pool[:limit]:
            line = f"- {r['name']} [{r.get('group', '')}] alone {r.get('auc_gain', 0):+.3f} ({r.get('years_helped', 0)}/4 years)"
            if r.get("team_size"):
                line += f", on the team {r['team_gain']:+.3f} ({r['team_years_helped']}/4)"
            if r.get("best_pair_partner"):
                line += f", best pair with {r['best_pair_partner']} {r['best_pair_gain']:+.3f}"
            lines.append(line + f" | {r['formula']}")
        return "\n".join(lines)

    def pairs_text(self, limit=12):
        if not self.pairs:
            return "(none yet)"
        lines = []
        for p in self.pairs[-limit:]:
            years = ", ".join(f"{y} {p.get(f'gain_{y}', float('nan')):+.3f}" for y in (2022, 2023, 2024, 2025))
            lines.append(f"- R{p['round']}: {p['a']} [{p['group_a']}] + {p['b']} [{p['group_b']}]: pair {p['gain']:+.3f} "
                         f"({years}; {p['years']}/4 years; LR {p['lr']:+.3f}, RF {p['rf']:+.3f}); alone on the team "
                         f"{p['a_alone_on_team']:+.3f} and {p['b_alone_on_team']:+.3f}, extra over the two added up "
                         f"{p['synergy']:+.3f} -> "
                         f"{'JOINED the team' if p['joined'] else 'not joined'}")
        return "\n".join(lines)

    # ---------------- one round ----------------
    def run_round(self, round_no):
        t0 = time.time()
        previous = [r["name"] for r in self.board if r["round"] == round_no - 1]
        prompt = build_prompt_v2(round_no=round_no, n_improve=self.n_improve, n_new=self.n_new,
                                 n_partner=self.n_partner, base_formulas=BASE_FORMULAS,
                                 stage1_top=self.stage1_top, market=self.market,
                                 notebook=lab_notebook(self.board), team_text=self.team_text(),
                                 promising_text=self.promising_text(), pairs_text=self.pairs_text(),
                                 rules=self.rules.as_dict(), previous_names=previous)
        tag = f"round_{round_no:02d}"
        (self.out_dir / f"{tag}_prompt.txt").write_text(prompt)
        response = self.proposer(prompt)
        (self.out_dir / f"{tag}_response.json").write_text(dump(response))

        for item in response.get("lessons", []) or []:          # Claude's lessons about the last round
            if not isinstance(item, dict):
                continue
            for r in self.board:
                if r["round"] == round_no - 1 and r["name"] == item.get("name"):
                    r["lesson"] = str(item.get("lesson", ""))[:300]

        n_total = self.n_improve + self.n_new + self.n_partner
        new = []
        for cand in response.get("factors", [])[:n_total]:
            r = self._build(cand, round_no)          # added one by one, so repeats within a round are caught too
            self.board.append(r)
            new.append(r)
        tested = [r for r in new if r["status"] == "tested"]
        for r in tested:                                          # 1. solo test
            res = self.judged(self.per_year(r["name"], ()), ())
            if not res["valid"]:
                r["status"], r["reason"] = "invalid", "constant or missing in a training/validation quarter"
                continue
            self.solo[r["name"]] = res
            self._write_solo(r, res)
            if self.curve is not None:                            # keep the uncurved result too, for reporting
                r["raw_gain"] = summarise_result(self.per_year(r["name"], ()))["gain"]
            r["max_corr"], r["closest"] = self.similarity(r["name"])
        tested = [r for r in tested if r["status"] == "tested"]
        team_before = len(self.team)
        self._grow_team(tested, how=f"team trial, round {round_no}")    # 2. team trial
        pairs_before = len(self.pairs)
        self._pair_trials(round_no, [r["name"] for r in tested if r["status"] != "kept"])   # 3. pair trials
        for r in self.board:                                       # 4. statuses (old promising ones may have joined)
            if r["status"] in ("tested", "promising"):
                self._set_status(r)
        for r in new:
            alone = f"{r['auc_gain']:+.4f} ({r['years_helped']}/4)" if "auc_gain" in r else "-"
            team = f"{r['team_gain']:+.4f} ({r['team_years_helped']}/4)" if r.get("team_size") else "-"
            print(f"  [{r['status']:9}] {r['name']}: alone {alone} | on team {team} "
                  f"| similarity {r.get('max_corr', float('nan')):.2f} | {r['reason']}", flush=True)
        new_pairs = self.pairs[pairs_before:]
        for p in new_pairs:
            print(f"  pair {p['a']} + {p['b']}: {p['gain']:+.4f} ({p['years']}/4), extra {p['synergy']:+.4f}"
                  f"{'  -> JOINED' if p['joined'] else ''}", flush=True)
        self.log.append({"round": round_no, "team_size": len(self.team), "joined": len(self.team) - team_before,
                         "promising": sum(r["status"] == "promising" for r in self.board),
                         "pairs_tried": len(new_pairs), "pairs_joined": sum(p["joined"] for p in new_pairs),
                         "mean_alone_gain": float(np.nanmean([r.get("auc_gain", np.nan) for r in tested])) if tested else np.nan,
                         "team": "|".join(self.team), "seconds": round(time.time() - t0, 1)})
        self.save()

    def finalize(self):
        """One more team trial for everything on the promising list, against the final team."""
        pool = [r for r in self.board if r["status"] == "promising"]
        before = len(self.team)
        self._grow_team(pool, how="final check of the promising list")
        for r in pool:
            self._set_status(r)
        self.log.append({"round": "final", "team_size": len(self.team), "joined": len(self.team) - before,
                         "promising": sum(r["status"] == "promising" for r in self.board),
                         "pairs_tried": 0, "pairs_joined": 0, "mean_alone_gain": np.nan,
                         "team": "|".join(self.team), "seconds": 0})
        self.save()

    # ---------------- saving and continuing ----------------
    def load_state(self) -> int:
        """Continue a stopped run from its saved files. Returns the next round number."""
        self.board = board_records(pd.read_csv(self.out_dir / "scoreboard.csv"))
        for r in self.board:
            if r["status"] == "invalid":
                continue
            self.values[r["name"]] = evaluate(r["formula"], self.prices)
            if "auc_gain" in r:
                self.solo[r["name"]] = {"valid": True, "gain": r["auc_gain"], "years": r["years_helped"],
                                        "lr": r["auc_gain_lr"], "rf": r["auc_gain_rf"],
                                        "by_year": {y: r[f"gain_{y}"] for y in (2022, 2023, 2024, 2025)}}
        try:
            self.pairs = pd.read_csv(self.out_dir / "pairs.csv").to_dict("records")
        except (FileNotFoundError, pd.errors.EmptyDataError):
            self.pairs = []
        self.log = pd.read_csv(self.out_dir / "rounds.csv").to_dict("records")
        self.team = [k["name"] for k in json.loads((self.out_dir / "kept_factors.json").read_text())]
        return int(max(r["round"] for r in self.board)) + 1

    def save(self):
        pd.DataFrame(self.board).to_csv(self.out_dir / "scoreboard.csv", index=False)
        pd.DataFrame(self.pairs).to_csv(self.out_dir / "pairs.csv", index=False)
        pd.DataFrame(self.log).to_csv(self.out_dir / "rounds.csv", index=False)
        kept = [{k: self.rec(n).get(k) for k in ("name", "formula", "hypothesis", "round", "how_selected",
                                                  "auc_gain", "years_helped", "team_gain")} for n in self.team]
        (self.out_dir / "kept_factors.json").write_text(json.dumps(kept, indent=2, default=float) + "\n")
        (self.out_dir / "rules.json").write_text(json.dumps(self.rules.as_dict(), indent=2) + "\n")


class TemplateProposerV2:
    """No-LLM stand-in for testing the loop offline: template formulas, modes and partners filled in."""

    def __init__(self, seed=0, n_improve=2, n_new=2, n_partner=2):
        self.base = RandomProposer(seed)
        self.modes = ["improve"] * n_improve + ["new"] * n_new + ["partner"] * n_partner

    def __call__(self, prompt):
        section = prompt.split("PROMISING LIST", 1)[1].split("PAIR TRIALS SO FAR", 1)[0]
        promising = re.findall(r"^- (\S+) \[", section, flags=re.M)
        out = []
        for i, mode in enumerate(self.modes):
            parent = promising[i % len(promising)] if (mode == "partner" and promising) else ""
            out.append({"name": f"template_{self.base.rng.randrange(10 ** 6)}", "mode": mode, "parent": parent,
                        "role": "", "formula": self.base._formula(), "hypothesis": "template formula (no LLM)"})
        return {"reflection": "template formulas (no LLM)", "lessons": [], "factors": out}


class ReplayProposer:
    """Replays the factors from an earlier run, round by round (for 'old clues, new judging')."""

    def __init__(self, run_dir):
        self.run_dir = Path(run_dir)

    def __call__(self, prompt):
        round_no = int(re.match(r"ROUND (\d+)", prompt).group(1))
        saved = json.loads((self.run_dir / f"round_{round_no:02d}_response.json").read_text())
        factors = [{**f, "mode": "improve" if f.get("mode") == "variant" else f.get("mode", "new"), "role": ""}
                   for f in saved.get("factors", [])]
        return {"reflection": f"replay of {self.run_dir.name}, round {round_no}", "lessons": [], "factors": factors}

## Step 6: Setting the New Bars with Random Numbers

Same idea as last week's fake-factor test: before any real run, I put factors made of **pure random numbers** through the new tests and counted how often they would get into the team by luck. The bars were fixed before the real runs and are the same for B and C. This check found three problems, and each one changed the design:

1. **Lucky pairs.** My first pair rule was "the pair must beat the better of its two members alone". A random factor that looks promising was a little lucky on its own. Put two lucky ones together and their luck simply adds up, so the pair always looks better than either one: 42% of promising random pairs passed. The rule now is: **the pair's gain must be clearly above the two gains added up.** Luck adds up; real teamwork gives more than the sum.
2. **The forest changed by itself.** With its default setting, the Random Forest looks at 4 factors at each split when it has up to 24 factors, but at 5 from 25 factors on. So on a team of 2, adding *any* extra column changed the forest itself, and 8 of 18 promising random factors passed the team trial. Now the forest always looks at 4 factors per split. For my 22 factors and for every single-factor test this is exactly what it did before, so last week's numbers do not change.
3. **Second chances.** I had planned to give every factor on the promising list one more team trial at the end. Even after fix 2, about 1 in 18 promising random factors got in that way, so this step is switched off.

In [ ]:
# Section 6: How often random numbers would get into the team
noise_team = pd.read_csv('results/stage2/noise_null_v2_team.csv', index_col=0)
noise_pairs = pd.read_csv('results/stage2/noise_null_v2_pairs.csv')
rules = json.loads(Path('results/stage2/rules_v2.json').read_text())
print('Bars used in runs B and C:', rules, '\n')

rows = []
for bar in [0.0, 0.005, rules['team_gain']]:
    share = ((noise_team.years >= 3) & (noise_team.gain >= bar)).mean()
    rows.append({'test': f'team trial ({len(noise_team)} random factors)',
                 'rule': f'helps 3+ of 4 years, gain >= {bar}', 'random factors getting in': f'{share:.0%}'})
for kind in ['random', 'promising']:
    d = noise_pairs[noise_pairs.kind == kind]
    first = ((d.years >= 3) & (d.gain >= 0.015) & (d.extra_vs_better >= 0.005)).mean()
    final = ((d.years >= 3) & (d.gain >= rules['pair_gain']) & (d.extra >= rules['pair_synergy'])).mean()
    rows.append({'test': f'pair trial ({len(d)} {kind} pairs)', 'rule': 'first rule: beats the better one alone by 0.005',
                 'random factors getting in': f'{first:.0%}'})
    rows.append({'test': f'pair trial ({len(d)} {kind} pairs)',
                 'rule': f"final rule: gain >= {rules['pair_gain']}, and {rules['pair_synergy']} above the two added up",
                 'random factors getting in': f'{final:.0%}'})
second = pd.read_csv('results/stage2/noise_null_v2_second_chance.csv')
share = ((second.years >= 3) & (second.gain >= rules['team_gain'])).mean()
rows.append({'test': f'second chance ({len(second)} promising random factors)',
             'rule': 'team trial again, on a 2-factor team', 'random factors getting in': f'{share:.0%}'})
display(pd.DataFrame(rows))

**Reading this output**

* **Team trial:** none of the 48 random factors would have joined the team. To join, a factor has to help in at least 3 of the 4 years with an average gain of at least 0.01.
* **Pair trial:** with my first rule ("beats the better one alone"), **19 of 64 (30%)** promising random pairs got in. With the final rule (pair gain at least 0.01, and at least 0.01 more than the two gains added up), **0 of 64** random pairs and **1 of 64** promising random pairs got in.
* **Second chance:** **1 of 18** promising random factors got in when they were tried again at the end, so this step is switched off.
* So a factor that joins the team in B or C has done something random numbers almost never do.

## Step 7: What Happened in the New Run (C)

8 rounds, 6 factors per round (48 in total, the same budget as last week).

In [ ]:
# Section 7a: Round by round: team size, promising list, pair trials
c_rounds = pd.read_csv(RUN_C / 'rounds.csv')
display(c_rounds[['round', 'team_size', 'joined', 'promising', 'pairs_tried', 'pairs_joined', 'mean_alone_gain', 'seconds']].round(4))

c_board = pd.read_csv(RUN_C / 'scoreboard.csv').fillna({'reason': '', 'lesson': '', 'how_selected': ''})
print('\nStatus of all factors tried:', c_board.status.value_counts().to_dict())

In [ ]:
# Section 7b: The selected team: each factor's formula, Claude's reason, and how it got in
c_team = json.loads((RUN_C / 'kept_factors.json').read_text())
if not c_team:
    print('No factor met the team or pair rules in this run.')
for k in c_team:
    r = c_board.set_index('name').loc[k['name']]
    years = ', '.join(f"{y}: {r[f'gain_{y}']:+.3f}" for y in [2022, 2023, 2024, 2025])
    print(f"{k['name']}  (round {k['round']}, {k['how_selected']})")
    print(f"  formula: {k['formula']}")
    print(f"  Claude's reason: {k['hypothesis']}")
    print(f"  alone, by year: {years}")
    print(f"  {r['reason']}\n")

In [ ]:
# Section 7c: All pair trials: the pair's gain, each one alone on the team, and the extra over the two added up
c_pairs = pd.read_csv(RUN_C / 'pairs.csv') if (RUN_C / 'pairs.csv').stat().st_size > 2 else pd.DataFrame()
if len(c_pairs):
    display(c_pairs[['round', 'a', 'b', 'group_a', 'group_b', 'gain', 'years', 'a_alone_on_team', 'b_alone_on_team', 'synergy', 'joined']]
            .rename(columns={'synergy': 'extra over the two added up'}).round(4))
else:
    print('No pair trials.')

**Reading this output**

* **No factor joined the team.** 14 ended on the promising list, 33 were rejected, 1 formula could not be used.
* Claude converged on **trading-volume factors** (Apple's volume surge compared with QQQ, NVIDIA or Microsoft). They gave the largest gains alone (up to +0.015) but only in 2 of 4 years, almost always 2023 and 2025.
* 34 pair trials: the best pairs reached +0.02, but their extra over the two gains added up stayed below the bar (at most +0.004).
* The run used about 97,000 input and 13,000 output tokens (well under US$1).

## Step 8: Last Week's Factors, Judged the New Way (B)

To separate the effect of the new **judging** from the effect of the new **feedback to Claude**, I took the same 48 factors Claude proposed last week, in the same order, and put them through the new team trial, pair trials and promising list. No new Claude call is needed for this.

In [ ]:
# Section 8: What the new rules pick from last week's 48 factors
b_board = pd.read_csv(RUN_B / 'scoreboard.csv').fillna({'reason': '', 'how_selected': ''})
b_team = json.loads((RUN_B / 'kept_factors.json').read_text())
print('Last week (A) kept:', [k['name'] for k in a_kept])
print('New rules (B) selected:')
for k in b_team:
    print(f"  {k['name']}  ({k['how_selected']})\n    {k['formula']}")
print('\nStatus of the 48 factors under the new rules:', b_board.status.value_counts().to_dict())

**Reading this output**

* Under the new rules, last week's 48 factors give a **team of 4** instead of 1 factor:
  * **nvda_range_vs_aapl_range**: last week's factor, joined in round 4
  * **msft_nvda_close_position_diff**: rejected last week (helped only 2 of 4 years alone), but on top of the team it helped **all 4 years** (+0.014). This is exactly the kind of factor Dr Liu said we were throwing away.
  * **a pair**, one Apple-relative factor (NVIDIA vs Apple volatility) and one market factor (NVIDIA range vs QQQ range): weak alone (-0.001 and +0.004 on the team), together **+0.013 in all 4 years, +0.010 more than the two added up**. This matches Dr Liu's example: a market-condition factor and an Apple-relative factor that help together.

## Step 9: Why So Few Factors Pass: Each Year Reacts the Same Way to Almost Any Factor

Looking at the report cards, Claude noticed it first (round 2 of run C): *"every factor helped in 2023 and 2025 but hurt in 2024"*. Below I count, for each year, how many factors helped: Claude's from both weeks, and the template formulas made with no LLM at all.

In [ ]:
# Section 9: Share of factors that helped in each year (checked on Q3)
def share_by_year(per_year):
    return per_year.groupby('year').apply(lambda d: (d.auc_gain > 0).mean())

a_years = pd.read_csv(RUN_A / 'scores_by_year.csv')
t_years = pd.read_csv('results/stage2/runs/20261015_1645_random/scores_by_year.csv')
gain_cols = [f'gain_{y}' for y in [2022, 2023, 2024, 2025]]
c_long = pd.read_csv(RUN_C / 'scoreboard.csv').query('status != "invalid"')[['name'] + gain_cols].melt(
    id_vars='name', value_vars=gain_cols, value_name='auc_gain')
c_long['year'] = c_long.variable.str[-4:].astype(int)
table = pd.DataFrame({'Claude, 15 Oct (48)': share_by_year(a_years[a_years.valid.astype(bool)]),
                      'Claude, 21 Oct (47)': share_by_year(c_long),
                      'Templates, no LLM (39)': share_by_year(t_years[t_years.valid.astype(bool)])})
display((table * 100).round(0).astype(int).astype(str) + '%')

**Reading this output**

* In **2023**, about 80-90% of all factors raised the score; in **2024**, only 2-12% did. This happens for Claude's factors in both weeks **and** for template formulas made with no LLM.
* So "helped in 3 of 4 years" partly measures the years, not the factors: a factor practically has to win 2022 or 2024, where almost nothing wins.
* The reason is the size of the check: one quarter of about 63 trading days per year, so luck is almost as large as the effects we are looking for.

## Step 10: A Longer Measuring Stick: 8 Checks Instead of 4

Each year has only one checking quarter (Q3, about 63 trading days), so luck is almost as large as the effects we are looking for. I added a second, earlier check every year: train on Q1, check on Q2. Every factor now gets **8 checks instead of 4**. The Q3 checks are exactly the same as before.

(I also tried "grading on a curve", comparing each factor with a typical template factor in the same year. Random numbers then looked better than real price factors in the bad years, so the bar had to rise to 0.02 and nothing passed. I did not use it.)

As before, the bar was set with random numbers and template formulas first, before looking at Claude's factors.

In [ ]:
# Section 10: The new bar, and Claude's factors measured with 8 checks
null8 = pd.read_csv('results/stage2/checks8_null.csv')
rules8 = json.loads(Path('results/stage2/rules_v2_8checks.json').read_text())
rows = []
for kind in ['noise', 'template']:
    x = null8[null8.kind == kind]
    for k in [6, 7]:
        rows.append({'factors': f'{kind} ({len(x)})', 'rule': f'helped >= {k} of 8 checks, gain >= {rules8["team_gain"]}',
                     'getting in': f"{((x.checks >= k) & (x.gain >= rules8['team_gain'])).mean():.0%}"})
display(pd.DataFrame(rows))
print('Bar used:', rules8['min_years'], 'of 8 checks and average gain >=', rules8['team_gain'])

real8 = pd.read_csv('results/stage2/checks8_real.csv').drop_duplicates('formula')
real8['week'] = real8.kind.map(lambda k: '15 Oct' if k.startswith('20261015') else '21 Oct')
print('\nChecks helped (out of 8): Claude factors vs random numbers')
display(pd.DataFrame({'Claude factors': real8.checks.value_counts().sort_index(),
                      'random numbers': null8[null8.kind == 'noise'].checks.value_counts().sort_index()}).fillna(0).astype(int))
print('Claude factors with the highest average gain:')
display(real8.sort_values('gain', ascending=False).head(8)[['week', 'factor', 'gain', 'checks', 'lr', 'rf']].round(4))

**Reading this output**

* With 8 checks the bar (set with random numbers first) is **at least 7 of 8 checks and an average gain of at least 0.0075**: 2% of random factors and 0% of template formulas pass.
* Only one family of Claude's factors is that steady: **a peer's daily range compared with Apple's** (QQQ vs Apple: 7 of 8; Microsoft vs Apple: 7 of 8; last week's NVIDIA vs Apple: 6 of 8).
* The **volume** factors from this week have the largest average gains (up to +0.022) but help in only 4-5 of 8 checks: strong in some periods, useless in others.
* Most of Claude's factors help in 4-5 of 8 checks, about the same as random numbers. Finding a factor that helps consistently is hard with this data.

## Step 11: Dr Liu's Suggestion: One Big Pile, Then Feature Selection

Dr Liu suggested (21 Oct email) generating a large set of factors with the LLM and then using feature selection to find a good combination. Every factor Claude has written so far goes into one pile, and **forward selection** picks the team: add the factor that improves the team most, if it passes the 8-check bar; re-test the rest on the bigger team; repeat. Then a few pairs are tried on top of the team.

In [ ]:
# Section 11: Forward selection on the whole pile
RUN_P = Path('results/stage2/pile_8checks')
p_team = json.loads((RUN_P / 'kept_factors.json').read_text())
p_log = pd.read_csv(RUN_P / 'selection_log.csv')
p_pairs = pd.read_csv(RUN_P / 'pairs.csv')
print(f"Pile: {len(pd.read_csv('results/stage2/checks8_real.csv').drop_duplicates('formula'))} different factors,",
      f"{len(pd.read_csv(RUN_P / 'scoreboard.csv'))} after removing near-copies of my 22 factors")
for k in p_team:
    print(f"Selected: {k['name']}  ({k['how_selected']})\n  {k['formula']}\n  Claude's reason: {k['hypothesis']}")
print('\nStep 2: best candidates on top of the selected factor')
display(p_log[p_log.step == 2].sort_values('gain', ascending=False).head(5).round(4))
print('Pair trials (gain, checks helped, and the extra over the two gains added up):')
display(p_pairs[['a', 'b', 'group_a', 'group_b', 'gain', 'checks', 'extra', 'joined']].round(4))

**Reading this output**

* From 81 different factors (after removing near-copies of my 22), forward selection picks **qqq_range_vs_aapl_range** (QQQ's daily range minus Apple's), helped 7 of 8 checks. No second factor passed the bar on top of it.
* The pair trials show real teamwork between a **market volume** factor (NVIDIA or QQQ volume surge) and an **Apple-relative volume** factor: together +0.017 to +0.020, which is **+0.005 to +0.009 more than the two added up**. But they helped in only 6 of 8 checks, one short of the bar, so they did not join. These are good parents for PARTNER requests in the next run.

## Step 12: The Comparison, and the Final Test on Q4

For each way of choosing factors I take the selected factors and do two things:
* **Check (Q3):** add all of them together to my 22 factors, and measure the gain in each year
* **Final test (Q4), used once per selected set:** train on Q1-Q2 with my 22 factors, and with the 22 plus the selected factors, predict Q4, and run the Stage 1 backtest (hold Apple only on days the model says "up")

All selections were finished before this test, and nothing is chosen based on it.

In [ ]:
# Section 12a: Check and final test for every way of choosing factors
scorer = Scorer(features)
summary, by_year = [], []
for label, kept in [('A: one at a time (15 Oct)', a_kept),
                    ('B: 15 Oct factors, team rules', json.loads((RUN_B / 'kept_factors.json').read_text())),
                    ('C: new loop with Claude (21 Oct)', c_team),
                    ('P: whole pile, 8 checks', p_team)]:
    row = {'run': label, 'selected': len(kept)}
    if kept:
        values = {k['name']: evaluate(k['formula'], prices) for k in kept}
        check = summarise_result(scorer.gains(values, {}))
        row.update({'check gain (Q3)': check['gain'], 'check years helped': check['years']})
        m, _ = run_final_test(features, prices, raw, kept)
        by_year.append(m.assign(run=label))
        for model, short in [('logistic_regression', 'LR'), ('random_forest', 'RF')]:
            d = m[m.model == model].pivot(index='year', columns='features')
            row[f'{short} test AUC change'] = (d['auc', 'stage1_plus_llm'] - d['auc', 'stage1_22']).mean()
            row[f'{short} test bal. acc. change'] = (d['balanced_accuracy', 'stage1_plus_llm'] - d['balanced_accuracy', 'stage1_22']).mean()
            row[f'{short} years better (AUC)'] = int((d['auc', 'stage1_plus_llm'] > d['auc', 'stage1_22']).sum())
    summary.append(row)
summary = pd.DataFrame(summary).set_index('run')
display(summary.T.round(4))

In [ ]:
# Section 12b: Test balanced accuracy in each year (0.5 = coin flip), and the backtest
final = pd.concat(by_year, ignore_index=True)
base = final[final.features == 'stage1_22'].drop_duplicates(['year', 'model']).assign(run='Stage 1 (22 factors)')
view = pd.concat([base, final[final.features == 'stage1_plus_llm']])
colors = {'Stage 1 (22 factors)': '#b9bec7', 'A: one at a time (15 Oct)': '#e08a2b',
          'B: 15 Oct factors, team rules': '#2f6db5', 'P: whole pile, 8 checks': '#3a9e6f'}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, model in zip(axes, ['logistic_regression', 'random_forest']):
    for run, col in colors.items():
        s = view[(view.model == model) & (view.run == run)].sort_values('year')
        if len(s):
            ax.plot(s.year, s.balanced_accuracy, marker='o', lw=2, color=col, label=run)
    ax.axhline(0.5, color='#999', lw=0.8, ls='--')
    ax.set_title(model.replace('_', ' ').title()); ax.set_xticks([2022, 2023, 2024, 2025])
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Test balanced accuracy'); axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

ret = view.pivot_table(index=['model', 'run'], columns='year', values='strategy_return').round(3)
ret.loc[('buy and hold', 'Apple'), :] = final.groupby('year').buy_hold_return.first().round(3)
print('Backtest return in Q4 of each year'); display(ret)

**Reading this output**

* **B (last week's factors, team rules) is the best**: compared with my 22 factors, Logistic Regression test AUC +0.040 (better in 3 of 4 years) and balanced accuracy +0.077; Random Forest balanced accuracy +0.038. Last week's single factor (A) gave +0.031, +0.065 and +0.010.
* In the backtest B also earned the most: on average per Q4, Logistic Regression +3.2% (my 22 factors: -1.2%, A: +2.1%) and Random Forest +6.5% (22 factors: +4.0%, A: +4.9%, buy-and-hold: +4.9%).
* **P (whole pile, 8 checks)** is the steadiest factor in the checks but gives smaller test gains (Logistic Regression balanced accuracy +0.033; Random Forest slightly worse, -0.015).
* **C** selected nothing, so it is the same as my 22 factors.
* To be careful: each test quarter is only 63 days, so these differences are small samples; the Random Forest AUC change for B is slightly negative (-0.003) even though its balanced accuracy improved.

## Step 13: Running the New Loop Live

Switched off so the notebook doesn't call Claude every time it is opened. To run it: set `RUN_LIVE = True` (and set the API key in Terminal before starting Jupyter). The same loop runs from Terminal with `python experiments/run_stage2_loop_v2.py --rounds 8` (add `--rules results/stage2/rules_v2_8checks.json` for 8 checks). Every prompt, reply and result is saved in a new folder in `results/stage2/runs_v2/`.

In [ ]:
# Section 13: Run the new loop live
RUN_LIVE = False                    # True = ask Claude for real (needs ANTHROPIC_API_KEY)
ROUNDS = 8

def run_loop_v2(rounds=ROUNDS, rules_file='results/stage2/rules_v2.json'):
    out = Path('results/stage2/runs_v2') / f'{time.strftime("%Y%m%d_%H%M")}_claude'
    r = json.loads(Path(rules_file).read_text())
    loop = LoopV2(proposer=ClaudeProposer('claude-sonnet-5-5', tool=TOOL_V2, system=SYSTEM_V2),
                  features=features, prices=prices, out_dir=out,
                  stage1_importance_path=Path('results/multi_year/permutation_importance_summary.csv'),
                  rules=Rules(**{**r, 'checks': tuple(r.get('checks', ['q3']))}))
    for i in range(1, rounds + 1):
        print(f'Round {i}')
        loop.run_round(i)
    print('Selected team:', loop.team, '| saved in', out)
    return loop

if RUN_LIVE:
    live = run_loop_v2()
else:
    print('RUN_LIVE is False: using the saved run', RUN_C.name)

## Summary

**What I built (Dr Liu, 20 Oct):** a report card for every factor, Claude's own lessons, a team trial, pair trials, partner requests, and a comparison with last week. All bars are set with random numbers before any real run.

**What worked**
* Testing factors **together** keeps factors that last week's rule threw away. On last week's factors it built a team of 4 (including one market + Apple-relative pair, Dr Liu's example) that beat last week's single factor on the final Q4 test for both models' balanced accuracy and in the backtest.
* Claude **learns from the report cards**: specific lessons, and it found the year problem by itself.

**What I learned about the data**
* Every year's check reacts to almost any factor in the same way (2023: almost all help; 2024: almost all hurt), because each check is only 63 days. That is why few factors pass, and why this week's Claude run found no team member.
* With 8 checks instead of 4, the steady factors are the **peer range vs Apple range** family; Claude's **volume** factors are strong but unsteady.
* Random-number checks caught four traps before they could fool us: lucky pairs adding up their luck, the forest changing at 25 factors, second chances, and grading on a curve.

**Next steps**
* Use 8 checks inside the loop, so Claude's report cards show the steadier picture
* Repeat each method a few times to see how stable the selected factors are
* Ask Claude for partners of the near-miss volume pairs (market volume + Apple-relative volume)
* Share the factor pile and the setting with the team, so others can try other feature-selection methods on the same factors